In [1]:
!pip -q install feast==0.64.0 pyarrow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires tenacity<10,

In [ ]:
import feast

print("Feast version:", feast.__version__)

Feast version: 0.64.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving cse_employability_skill_gap_dataset.csv to cse_employability_skill_gap_dataset.csv


In [ ]:
import pandas as pd

df = pd.read_csv("/content/cse_employability_skill_gap_dataset.csv")

print(df.head())
print(df.shape)

  Student_ID  Programming  Databases  Problem_Solving  Communication  \
0       S001           73         82               63             56   
1       S002           86         49               70             45   
2       S003           63         42               47             82   
3       S004           49         48               66             50   
4       S005           77         57               41             67   

   Cloud_Computing  Teamwork  Aptitude  Average_Skill  Average_Skill_Gap  \
0               55        60        95          69.14              10.43   
1               82        69        73          67.71              11.29   
2               54        84        39          58.71              20.00   
3               90        59        56          59.71              19.57   
4               42        58        63          57.86              18.57   

  Skill_Gap_Category  Industry_Readiness_Percent  
0             Medium                       89.57  
1       

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print(df.columns)
print(df.info())
print(df.isnull().sum())

Index(['Student_ID', 'Programming', 'Databases', 'Problem_Solving',
       'Communication', 'Cloud_Computing', 'Teamwork', 'Aptitude',
       'Average_Skill', 'Average_Skill_Gap', 'Skill_Gap_Category',
       'Industry_Readiness_Percent'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Student_ID                  100 non-null    object 
 1   Programming                 100 non-null    int64  
 2   Databases                   100 non-null    int64  
 3   Problem_Solving             100 non-null    int64  
 4   Communication               100 non-null    int64  
 5   Cloud_Computing             100 non-null    int64  
 6   Teamwork                    100 non-null    int64  
 7   Aptitude                    100 non-null    int64  
 8   Average_Skill               100 non-null    float64
 9   Av

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("cse_employability_skill_gap_dataset.csv")

# Entity
df["student_id"] = df["Student_ID"].astype("string")

# Handle missing values
skill_columns = [
    "Programming",
    "Databases",
    "Problem_Solving",
    "Communication",
    "Cloud_Computing",
    "Teamwork",
    "Aptitude"
]

for column in skill_columns:
    df[column] = df[column].fillna(df[column].median())

# Encode Skill Gap Category
df["skill_gap_encoded"] = df["Skill_Gap_Category"].map({
    "Low": 0,
    "Medium": 1,
    "High": 2
}).astype("int64")

# Derived feature 1
# Average of all technical and employability skills
df["average_skill"] = (
    df["Programming"] +
    df["Databases"] +
    df["Problem_Solving"] +
    df["Communication"] +
    df["Cloud_Computing"] +
    df["Teamwork"] +
    df["Aptitude"]
) / 7

df["average_skill"] = df["average_skill"].astype("float32")

# Derived feature 2
# Overall industry readiness
df["industry_readiness"] = (
    df["Industry_Readiness_Percent"]
).astype("float32")

# Rename/cast other features
df["programming"] = df["Programming"].astype("int64")
df["databases"] = df["Databases"].astype("int64")
df["problem_solving"] = df["Problem_Solving"].astype("int64")
df["communication"] = df["Communication"].astype("int64")
df["cloud_computing"] = df["Cloud_Computing"].astype("int64")
df["teamwork"] = df["Teamwork"].astype("int64")
df["aptitude"] = df["Aptitude"].astype("int64")

# Target
df["skill_gap_category"] = df["Skill_Gap_Category"].astype("string")

# Display dataset
print(df.head())

  Student_ID  Programming  Databases  Problem_Solving  Communication  \
0       S001           73         82               63             56   
1       S002           86         49               70             45   
2       S003           63         42               47             82   
3       S004           49         48               66             50   
4       S005           77         57               41             67   

   Cloud_Computing  Teamwork  Aptitude  Average_Skill  Average_Skill_Gap  ...  \
0               55        60        95          69.14              10.43  ...   
1               82        69        73          67.71              11.29  ...   
2               54        84        39          58.71              20.00  ...   
3               90        59        56          59.71              19.57  ...   
4               42        58        63          57.86              18.57  ...   

  average_skill  industry_readiness programming  databases  problem_solving  \
0

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
df[
    [
        "student_id",
        "programming",
        "databases",
        "problem_solving",
        "communication",
        "cloud_computing",
        "teamwork",
        "aptitude",
        "skill_gap_category"
    ]
].head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,student_id,programming,databases,problem_solving,communication,cloud_computing,teamwork,aptitude,skill_gap_category
0,S001,73,82,63,56,55,60,95,Medium
1,S002,86,49,70,45,82,69,73,Medium
2,S003,63,42,47,82,54,84,39,Medium
3,S004,49,48,66,50,90,59,56,Medium
4,S005,77,57,41,67,42,58,63,Medium


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
base_time = pd.Timestamp(
    "2026-01-01",
    tz="UTC"
)


df["event_timestamp"] = (
    base_time +
    pd.to_timedelta(
        df["student_id"].str[1:].astype(int),
        unit="s"
    )
)


df["created_timestamp"] = (
    df["event_timestamp"] +
    pd.Timedelta(seconds=1)
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
feature_df = df[
    [
        "student_id",
        "event_timestamp",
        "created_timestamp",
        "programming",
        "databases",
        "problem_solving",
        "communication",
        "cloud_computing",
        "teamwork",
        "aptitude"
    ]
].copy()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
label_df = df[
    [
        "student_id",
        "event_timestamp",
        "skill_gap_category"
    ]
].copy()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print("Feature data:")
display(feature_df.head())


print("Labels:")
display(label_df.head())

Feature data:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,student_id,event_timestamp,created_timestamp,programming,databases,problem_solving,communication,cloud_computing,teamwork,aptitude
0,S001,2026-01-01 00:00:01+00:00,2026-01-01 00:00:02+00:00,73,82,63,56,55,60,95
1,S002,2026-01-01 00:00:02+00:00,2026-01-01 00:00:03+00:00,86,49,70,45,82,69,73
2,S003,2026-01-01 00:00:03+00:00,2026-01-01 00:00:04+00:00,63,42,47,82,54,84,39
3,S004,2026-01-01 00:00:04+00:00,2026-01-01 00:00:05+00:00,49,48,66,50,90,59,56
4,S005,2026-01-01 00:00:05+00:00,2026-01-01 00:00:06+00:00,77,57,41,67,42,58,63


Labels:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,student_id,event_timestamp,skill_gap_category
0,S001,2026-01-01 00:00:01+00:00,Medium
1,S002,2026-01-01 00:00:02+00:00,Medium
2,S003,2026-01-01 00:00:03+00:00,Medium
3,S004,2026-01-01 00:00:04+00:00,Medium
4,S005,2026-01-01 00:00:05+00:00,Medium


In [ ]:
import os


repo_path = "/content/cse_employability_feast"


os.makedirs(
    f"{repo_path}/data",
    exist_ok=True
)

In [ ]:
feature_df.to_parquet(
    f"{repo_path}/data/cse_employability_features.parquet",
    index=False
)

In [ ]:
feature_store_yaml = """
project: cse_employability_project


registry: data/registry.db


provider: local


offline_store:
  type: file


online_store:
  type: sqlite
  path: data/online_store.db
"""


with open(
    f"{repo_path}/feature_store.yaml",
    "w"
) as f:
    f.write(feature_store_yaml)

In [ ]:
feature_definition = '''
from datetime import timedelta

from feast import (
Entity,
FeatureView,
FeatureService,
Field,
FileSource
)

from feast.types import (
Int64
)

# -----------------------------

# ENTITY

# -----------------------------

student = Entity(
name="student",
join_keys=["student_id"],
description="CSE student"
)

# -----------------------------

# DATA SOURCE

# -----------------------------

cse_source = FileSource(
name="cse_source",
path="data/cse_employability_features.parquet",
timestamp_field="event_timestamp",
created_timestamp_column="created_timestamp"
)

# -----------------------------

# FEATURE VIEW

# -----------------------------

cse_feature_view = FeatureView(
name="cse_employability_features",
entities=[student],

ttl=timedelta(days=50000),

schema=[
    Field(name="programming", dtype=Int64),
    Field(name="databases", dtype=Int64),
    Field(name="problem_solving", dtype=Int64),
    Field(name="communication", dtype=Int64),
    Field(name="cloud_computing", dtype=Int64),
    Field(name="teamwork", dtype=Int64),
    Field(name="aptitude", dtype=Int64),
],

source=cse_source,

online=True

)

# -----------------------------

# FEATURE SERVICE

# -----------------------------

cse_feature_service = FeatureService(
name="cse_employability_service",
features=[
cse_feature_view
]
)
'''

with open(
f"{repo_path}/features.py",
"w"
) as f:
    f.write(feature_definition)

In [ ]:
!find /content/cse_employability_feast -maxdepth 2 -type f

/content/cse_employability_feast/feature_store.yaml
/content/cse_employability_feast/data/cse_employability_features.parquet
/content/cse_employability_feast/features.py


In [ ]:
%cd /content/cse_employability_feast

/content/cse_employability_feast


In [ ]:
!feast apply

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [ ]:
!feast entities list

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [ ]:
!feast feature-views list

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [ ]:
from feast import FeatureStore


store = FeatureStore(
    repo_path="/content/cse_employability_feast"
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
feature_service = store.get_feature_service(
    "cse_employability_service"
)

In [ ]:
entity_df = label_df.copy()


display(entity_df.head())

,student_id,event_timestamp,skill_gap_category
0,S001,2026-01-01 00:00:01+00:00,Medium
1,S002,2026-01-01 00:00:02+00:00,Medium
2,S003,2026-01-01 00:00:03+00:00,Medium
3,S004,2026-01-01 00:00:04+00:00,Medium
4,S005,2026-01-01 00:00:05+00:00,Medium


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
training_data = store.get_historical_features(
    entity_df=entity_df,
    features=feature_service
).to_df()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
display(training_data.head())

,student_id,event_timestamp,skill_gap_category,programming,databases,problem_solving,communication,cloud_computing,teamwork,aptitude
0,S001,2026-01-01 00:00:01+00:00,Medium,73,82,63,56,55,60,95
1,S002,2026-01-01 00:00:02+00:00,Medium,86,49,70,45,82,69,73
2,S003,2026-01-01 00:00:03+00:00,Medium,63,42,47,82,54,84,39
3,S004,2026-01-01 00:00:04+00:00,Medium,49,48,66,50,90,59,56
4,S005,2026-01-01 00:00:05+00:00,Medium,77,57,41,67,42,58,63


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
feature_columns = [
    "programming",
    "databases",
    "problem_solving",
    "communication",
    "cloud_computing",
    "teamwork",
    "aptitude"
]

In [ ]:
X = training_data[feature_columns]


y = training_data["skill_gap_category"]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.tree import DecisionTreeClassifier


model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)


model.fit(
    X_train,
    y_train
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DecisionTreeClassifier(max_depth=4, random_state=42)

In [ ]:
predictions = model.predict(X_test)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.metrics import accuracy_score


accuracy = accuracy_score(
    y_test,
    predictions
)


print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

Accuracy: 80.0 %


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
%cd /content/cse_employability_feast


!feast materialize \
    2026-01-01T00:00:00 \
    2026-01-02T00:00:00

/content/cse_employability_feast
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/

In [ ]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {
            "student_id": 25
        }
    ]
).to_dict()

In [ ]:
print(online_features)

{'student_id': ['25'], 'teamwork': [None], 'cloud_computing': [None], 'communication': [None], 'aptitude': [None], 'programming': [None], 'databases': [None], 'problem_solving': [None]}


In [ ]:
online_df = pd.DataFrame(
    online_features
)


display(online_df)

,student_id,teamwork,cloud_computing,communication,aptitude,programming,databases,problem_solving
0,25,None,None,None,None,None,None,None


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
final_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

final_model.fit(X, y)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DecisionTreeClassifier(max_depth=4, random_state=42)

In [ ]:
X_online = online_df[
    feature_columns
]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
prediction = final_model.predict(
    X_online
)


print(
    "Predicted skill gap:",
    prediction[0]
)

Predicted skill gap: Medium


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"student_id": 10},
        {"student_id": 20},
        {"student_id": 30},
        {"student_id": 40}
    ]
).to_dict()


online_df = pd.DataFrame(
    online_features
)


display(online_df)

,student_id,teamwork,cloud_computing,communication,aptitude,programming,databases,problem_solving
0,10,None,None,None,None,None,None,None
1,20,None,None,None,None,None,None,None
2,30,None,None,None,None,None,None,None
3,40,None,None,None,None,None,None,None


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
online_df["predicted_skill_gap"] = (
    final_model.predict(
        online_df[feature_columns]
    )
)


display(
    online_df[
        [
            "student_id",
            "predicted_skill_gap"
        ]
    ]
)

,student_id,predicted_skill_gap
0,10,Medium
1,20,Medium
2,30,Medium
3,40,Medium


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
